### Setup + Autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import pyro
import pyro.infer as infer
from pyro.infer import MCMC, NUTS
import numpy as np

# Import our validated modules
from src.model import PhylogeneticPrior
from src.penalty import get_fresh_test_quartets
from src.diagnostics import evaluate_test_diagnostics, print_diagnostic_report, compute_nj_residual

# Enforce reproducibility
pyro.set_rng_seed(101)

: 

### Test Run (N=15)

In [ ]:
N_pilot = 15          # Scaled down from 50 for quick validation
K_pilot = 2           # Hyperbolic latent dimension
B_train = 100         # Training quartet subset budget
B_test = 200          # Evaluation test quartet budget
TAU = 0.05            # Temperature parameter for softplus relaxation

print(f"🚀 Initializing Pilot Run: N={N_pilot}, K={K_pilot}, B_train={B_train}")

# Generate fixed training structure and a clean validation set
prior_model = PhylogeneticPrior(N=N_pilot, K=K_pilot, B=B_train, seed=42)
test_quartets = get_fresh_test_quartets(N=N_pilot, B_test=B_test, train_quartets=prior_model.fixed_indices, seed=123)

# Define our calibration grid for lambda_4
lambda_grid = [0.0, 0.1, 1.0, 10.0, 50.0]
results_archive = {}

for lmbda in lambda_grid:
    print("\n" + "#"*60)
    print(f" λ4 = {lmbda} | Model Calibration Step")
    print("#"*60)
    
    # Wrapper function for Pyro matching NUTS signature
    def model_conditioned():
        return prior_model.initialize(lmbda_4=lmbda, lmbda_g=0.0, tau=TAU, use_scale=False)
    
    # Clear parameter store for clean sampling paths
    pyro.clear_param_store()
    
    # Configure NUTS kernel
    nuts_kernel = NUTS(model_conditioned, adapt_step_size=True, target_accept_prob=0.85)
    
    # Define execution container for MCMC sampling
    mcmc = MCMC(
        nuts_kernel,
        num_samples=200,      # Small sample count for quick feedback
        warmup_steps=100,      # Fast adaptation
        num_chains=1
    )
    
    # Execute chain calculation
    mcmc.run()

    # 1. Extract the raw posterior sample distributions for unconstrained variables (e.g., 'u')
    posterior_samples = mcmc.get_samples()
    
    # 2. Re-run the model graph over those sample traces to extract deterministic nodes
    predictive = infer.Predictive(model_conditioned, posterior_samples=posterior_samples)
    predictive_samples = predictive()
    
    # 3. Safely pull out your target scale-normalized distance matrices
    D_samples = predictive_samples["D_tilde"]  # Shape: (S, N, N)
    
    # 4. RUN POST-HOC DIAGNOSTICS
    print(f"\n📊 Processing diagnostics across {D_samples.shape[0]} posterior samples...")
    metrics = evaluate_test_diagnostics(D_samples, test_quartets, epsilon=0.05, gamma=0.05)
    print_diagnostic_report(metrics, epsilon=0.05, gamma=0.05)
    
    # Calculate post-hoc tree projection residuals (NJ Residuals)
    nj_residuals = [compute_nj_residual(D_samples[s]) for s in range(D_samples.shape[0])]
    mean_nj_residual = np.mean(nj_residuals)
    print(f"  └─ Post-hoc Mean NJ Tree Residual R_NJ(D~): {mean_nj_residual * 100:.3f}%")
    
    # Store summaries to isolate optimal landscape settings
    results_archive[lmbda] = {
        "metrics": metrics,
        "mean_nj_residual": mean_nj_residual,
        "divergences": mcmc.diagnostics().get("divergences", {}).get("chain 0", 0)
    }


print("\n" + "="*60)
print("🏁 PILOT CALIBRATION COMPLETE — SUMMARY FINDINGS")
print("="*60)
for lmbda, res in results_archive.items():
    q95 = res["metrics"]["hard_violations"]["quantile_95"]
    nj_pct = res["mean_nj_residual"] * 100
    divs = res["divergences"]
    print(f"λ4 = {lmbda:5.1f} ── q_0.95 Violations: {q95:.4f} ── NJ Residual: {nj_pct:.2f}% ── Divergences: {divs}")

: 